In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Устанавливаем skopeo и zstd без лишнего мусора в логах
!apt-get update -qq && apt-get install skopeo zstd -y -qq

# Создаем изолированные временные папки, чтобы не забивать системный корень Linux
!mkdir -p /content/skopeo_cache
mountaineer_dir = "/content/triton_oci"
!mkdir -p $mountaineer_dir

# 2. Скачиваем образ Triton Server, принудительно перенаправив временный кэш на быстрый диск
print("🚀 [1/3] Начинаем скачивание слоев NVIDIA Triton в локальную OCI-папку...")
# Флаг --override-arch гарантирует скачивание amd64, даже если Colab запустился на ARM
!skopeo --tmpdir=/content/skopeo_cache copy --override-arch amd64 docker://nvcr.io/nvidia/tritonserver:25.12-py3 oci:$mountaineer_dir:latest

# 3. Упаковываем локально на высокой скорости с безопасным уровнем сжатия (Защита от OOM Killer)
print("📦 [2/3] Локально сжимаем слои через zstd (Безопасный буфер ОЗУ)...")
local_archive = "/content/tritonserver_25.12-py3.tar.zst"
# Уровень -3 — это промышленный стандарт: высокая скорость, минимальное потребление RAM ноутбука/Colab
!tar -cf - -C $mountaineer_dir . | zstd -3 > $local_archive

# Проверяем, что файл реально создался локально и имеет вес
import os
if os.path.exists(local_archive) and os.path.getsize(local_archive) > 0:
    size_gb = os.path.getsize(local_archive) / (1024**3)
    print(f"📊 Локальный архив успешно собран! Размер сжатого файла: {size_gb:.2f} GB")

    # 4. Безопасное копирование готового файла на Google Диск (Защита от сетевых обрывов)
    print("📥 [3/3] Безопасный перенос готового архива на ваш Google Диск...")
    !cp $local_archive /content/drive/MyDrive/

    # Верификация наличия файла на Google Диске
    if os.path.exists("/content/drive/MyDrive/tritonserver_25.12-py3.tar.zst"):
        print("🎉 [SUCCESS] Файл 'tritonserver_25.12-py3.tar.zst' успешно зафиксирован и сохранен на вашем Google Диске!")
    else:
        print("❌ Ошибка: Сетевой мост Google Диска отклонил сохранение файла. Проверьте свободное место на Диске.")
else:
    print("❌ Критическая ошибка: Файл не был собран локально из-за нехватки диска в самом Colab.")

# Очищаем за собой гигабайты временного мусора, чтобы вернуть место инстансу
!rm -rf /content/skopeo_cache $mountaineer_dir $local_archive 2>/dev/null


Streaming output truncated to the last 5000 lines.
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…

# Распаковываем архив напрямую в Docker хоста из папки загрузок Windows
zstd -d -c /mnt/c/Users/alexfil/Downloads/tritonserver_25.12-py3.tar.zst | docker load

docker tag 9c15ecb62070 nvcr.io/nvidia/tritonserver:25.12-py3

In [3]:
# 1. Устанавливаем skopeo и zstd без лишнего мусора в логах
!apt-get update -qq && apt-get install skopeo zstd -y -qq

# Создаем изолированные временные папки, чтобы не забивать системный корень Linux
!mkdir -p /content/skopeo_cache
mountaineer_dir = "/content/triton_oci"
!mkdir -p $mountaineer_dir

# 2. Скачиваем образ Triton Server, принудительно перенаправив временный кэш на быстрый диск
print("🚀 [1/3] Начинаем скачивание слоев NVIDIA Triton в локальную OCI-папку...")
# Флаг --override-arch гарантирует скачивание amd64, даже если Colab запустился на ARM
!skopeo --tmpdir=/content/skopeo_cache copy --override-arch amd64 docker://nvcr.io/nvidia/tritonserver:25.12-py3-sdk oci:$mountaineer_dir:latest

# 3. Упаковываем локально на высокой скорости с безопасным уровнем сжатия (Защита от OOM Killer)
print("📦 [2/3] Локально сжимаем слои через zstd (Безопасный буфер ОЗУ)...")
local_archive = "/content/tritonserver_25.12-py3-sdk.tar.zst"
# Уровень -3 — это промышленный стандарт: высокая скорость, минимальное потребление RAM ноутбука/Colab
!tar -cf - -C $mountaineer_dir . | zstd -3 > $local_archive

# Проверяем, что файл реально создался локально и имеет вес
import os
if os.path.exists(local_archive) and os.path.getsize(local_archive) > 0:
    size_gb = os.path.getsize(local_archive) / (1024**3)
    print(f"📊 Локальный архив успешно собран! Размер сжатого файла: {size_gb:.2f} GB")

    # 4. Безопасное копирование готового файла на Google Диск (Защита от сетевых обрывов)
    print("📥 [3/3] Безопасный перенос готового архива на ваш Google Диск...")
    !cp $local_archive /content/drive/MyDrive/

    # Верификация наличия файла на Google Диске
    if os.path.exists("/content/drive/MyDrive/tritonserver_25.12-py3-sdk.tar.zst"):
        print("🎉 [SUCCESS] Файл 'tritonserver_25.12-py3-sdk.tar.zst' успешно зафиксирован и сохранен на вашем Google Диске!")
    else:
        print("❌ Ошибка: Сетевой мост Google Диска отклонил сохранение файла. Проверьте свободное место на Диске.")
else:
    print("❌ Критическая ошибка: Файл не был собран локально из-за нехватки диска в самом Colab.")

# Очищаем за собой гигабайты временного мусора, чтобы вернуть место инстансу
!rm -rf /content/skopeo_cache $mountaineer_dir $local_archive 2>/dev/null


Streaming output truncated to the last 5000 lines.
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…
…

# Распаковываем архив напрямую в Docker хоста из папки загрузок Windows
zstd -d -c /mnt/c/Users/alexfil/Downloads/tritonserver_25.12-py3-sdk.tar.zst | docker load

docker tag bb88844cdb33 nvcr.io/nvidia/tritonserver:25.12-py3-sdk